<a href="https://colab.research.google.com/github/intellix-tcc/Intellix/blob/main/Treinamento_Modelo_Intellix_Parte_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mv /gerar_dataset.py /moldes.py /validar_dataset.py /content/
%cd /content
!ls

/content
gerar_dataset.py  moldes.py  sample_data  validar_dataset.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python gerar_dataset.py


✅ 4800 exemplos únicos gerados em dados/dataset.jsonl

Exemplos por intenção (o ideal é estarem parecidos entre si):
  comparacao_periodos       400  ████████████████████████████████████████
  desempenho_vendedor       400  ████████████████████████████████████████
  produtos_sem_saida        400  ████████████████████████████████████████
  quantidade_vendas         400  ████████████████████████████████████████
  ticket_medio              400  ████████████████████████████████████████
  top_clientes              400  ████████████████████████████████████████
  top_produtos              400  ████████████████████████████████████████
  total_vendas_periodo      400  ████████████████████████████████████████
  variacao_periodo          400  ████████████████████████████████████████
  vendas_por_canal          400  ████████████████████████████████████████
  vendas_por_categoria      400  ████████████████████████████████████████
  vendas_por_pagamento      400  ███████████████████████████████████

In [ ]:
!python validar_dataset.py dados/dataset.jsonl


Validando: dados/dataset.jsonl
Exemplos: 4800
Intenções: 12/12
Tags usadas: 19/19
Tamanho das frases: min 2 | média 7.2 | max 14

✅ Dataset válido. Pode treinar.


In [ ]:
!head -3 dados/dataset.jsonl

{"texto": "faturamento de agosto", "intencao": "total_vendas_periodo", "tokens": ["faturamento", "de", "agosto"], "tags": ["O", "O", "B-PERIODO"], "origem": "molde"}
{"texto": "de quanto foi a queda em maio", "intencao": "variacao_periodo", "tokens": ["de", "quanto", "foi", "a", "queda", "em", "maio"], "tags": ["O", "O", "O", "O", "O", "O", "B-PERIODO"], "origem": "molde"}
{"texto": "quanto vendi março", "intencao": "total_vendas_periodo", "tokens": ["quanto", "vendi", "março"], "tags": ["O", "O", "B-PERIODO"], "origem": "molde"}


In [3]:
!mv /gerar_dataset.py /moldes.py /validar_dataset.py /content/
%cd /content
!ls

mv: cannot stat '/gerar_dataset.py': No such file or directory
mv: cannot stat '/moldes.py': No such file or directory
mv: cannot stat '/validar_dataset.py': No such file or directory
/content
sample_data


In [ ]:
import json, re
from pathlib import Path
from gerar_dataset import tokenizar
from moldes import VALORES, INTENCOES

FRASES = """
total_vendas_periodo | oi, quanto que eu vendi mês passado?
total_vendas_periodo | me diz aí o total que entrou em março
top_produtos         | qual foi o produto que mais saiu em abril
ticket_medio         | quanto em média cada cliente deixa na loja
comparacao_periodos  | foi melhor em março ou em abril
desempenho_vendedor  | a fernanda tá vendendo bem esse mês?
vendas_por_pagamento | o povo tá pagando mais no pix ou no cartão de crédito
produtos_sem_saida   | tem alguma coisa encalhada no estoque
quantidade_vendas    | fiz quantas vendas hoje
vendas_por_canal     | o instagram tá trazendo venda?
top_clientes         | quem mais gastou aqui esse ano
vendas_por_categoria | como tá a saída de eletrônicos
variacao_periodo     | eu cresci ou caí em relação ao mês passado
"""

CATALOGO = sorted(
    [(tokenizar(v), t.upper()) for t, vs in VALORES.items() for v in vs if tokenizar(v)],
    key=lambda x: -len(x[0]))

def pre_etiquetar(frase):
    tokens = tokenizar(frase)
    tags = ["O"] * len(tokens)
    i = 0
    while i < len(tokens):
        for vt, tipo in CATALOGO:
            n = len(vt)
            if tokens[i:i+n] == vt and all(t == "O" for t in tags[i:i+n]):
                tags[i] = f"B-{tipo}"
                for k in range(1, n):
                    tags[i+k] = f"I-{tipo}"
                i += n
                break
        else:
            if re.fullmatch(r"\d+", tokens[i]):
                tags[i] = "B-NUMERO"
            i += 1
    return tokens, tags

saida = []
for linha in FRASES.strip().splitlines():
    intencao, frase = (p.strip() for p in linha.split("|", 1))
    assert intencao in INTENCOES, f"intenção inválida: {intencao}"
    tokens, tags = pre_etiquetar(frase)
    saida.append({"texto": " ".join(tokens), "intencao": intencao,
                  "tokens": tokens, "tags": tags, "origem": "manual"})

Path("dados").mkdir(exist_ok=True)
with open("dados/teste_manual.jsonl", "w", encoding="utf-8") as f:
    for ex in saida:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"✅ {len(saida)} frases etiquetadas em dados/teste_manual.jsonl\n")
for ex in saida[:3]:
    print(f"→ {ex['intencao']}")
    for tok, tag in zip(ex["tokens"], ex["tags"]):
        print(f"   {'🏷️' if tag != 'O' else '  '} {tok:<16} {tag}")
    print()

✅ 13 frases etiquetadas em dados/teste_manual.jsonl

→ total_vendas_periodo
      oi               O
      ,                O
      quanto           O
      que              O
      eu               O
      vendi            O
   🏷️ mês              B-PERIODO
   🏷️ passado          I-PERIODO
      ?                O

→ total_vendas_periodo
      me               O
      diz              O
      aí               O
      o                O
      total            O
      que              O
      entrou           O
      em               O
   🏷️ março            B-PERIODO

→ top_produtos
      qual             O
      foi              O
      o                O
      produto          O
      que              O
      mais             O
      saiu             O
      em               O
   🏷️ abril            B-PERIODO



In [ ]:
from google.colab import files
files.download("dados/dataset.jsonl")
files.download("dados/teste_manual.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import json, re
from pathlib import Path
from gerar_dataset import tokenizar
from moldes import VALORES, INTENCOES

FRASES = """
total_vendas_periodo | oi, quanto que eu vendi mês passado?
total_vendas_periodo | me diz aí o total que entrou em março
top_produtos         | qual foi o produto que mais saiu em abril
ticket_medio         | quanto em média cada cliente deixa na loja
comparacao_periodos  | foi melhor em março ou em abril
desempenho_vendedor  | a fernanda tá vendendo bem esse mês?
vendas_por_pagamento | o povo tá pagando mais no pix ou no cartão de crédito
produtos_sem_saida   | tem alguma coisa encalhada no estoque
quantidade_vendas    | fiz quantas vendas hoje
vendas_por_canal     | o instagram tá trazendo venda?
top_clientes         | quem mais gastou aqui esse ano
vendas_por_categoria | como tá a saída de eletrônicos
variacao_periodo     | eu cresci ou caí em relação ao mês passado
"""

CATALOGO = sorted(
    [(tokenizar(v), t.upper()) for t, vs in VALORES.items() for v in vs if tokenizar(v)],
    key=lambda x: -len(x[0]))

def pre_etiquetar(frase):
    tokens = tokenizar(frase)
    tags = ["O"] * len(tokens)
    i = 0
    while i < len(tokens):
        for vt, tipo in CATALOGO:
            n = len(vt)
            if tokens[i:i+n] == vt and all(t == "O" for t in tags[i:i+n]):
                tags[i] = f"B-{tipo}"
                for k in range(1, n):
                    tags[i+k] = f"I-{tipo}"
                i += n
                break
        else:
            if re.fullmatch(r"\d+", tokens[i]):
                tags[i] = "B-NUMERO"
            i += 1
    return tokens, tags

saida = []
for linha in FRASES.strip().splitlines():
    intencao, frase = (p.strip() for p in linha.split("|", 1))
    assert intencao in INTENCOES, f"intenção inválida: {intencao}"
    tokens, tags = pre_etiquetar(frase)
    saida.append({"texto": " ".join(tokens), "intencao": intencao,
                  "tokens": tokens, "tags": tags, "origem": "manual"})

Path("dados").mkdir(exist_ok=True)
with open("dados/teste_manual.jsonl", "w", encoding="utf-8") as f:
    for ex in saida:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"✅ {len(saida)} frases criadas em dados/teste_manual.jsonl")

ModuleNotFoundError: No module named 'gerar_dataset'